# Reto: COVID-19 CT Image Segmentation
## Pipeline de Extracción, Limpieza y Transformación de Datos (ETL)

**Objetivo del Reto:**
Segmentación semántica de lesiones pulmonares provocadas por COVID-19 en tomografías computarizadas (CT) axiales, específicamente:
- **Clase 0:** Vidrio Deslustrado (*Ground-Glass Opacity - GGO*)
- **Clase 1:** Consolidaciones pulmonares (*Consolidations*)
- **Métrica de Evaluación:** Pixel-wise Macro F1-Score sobre las 10 imágenes de prueba (`test_images_medseg.npy`).

---
### Contenido del Notebook:
1. **Extracción (Extract):** Carga y validación dimensional de arreglos NumPy (`Medseg`, `Radiopedia`, `Test`).
2. **Diagnóstico Exploratorio:** Identificación de inconsistencias, colisiones de máscaras y rango de unidades Hounsfield (HU).
3. **Limpieza de Datos (Data Cleaning):**
   - Validación de nulos (NaNs) e infinitos (Infs).
   - Corrección del solapamiento multiclase anómalo entre lesiones y fondo (510 px en Medseg, 22,027 px en Radiopedia).
   - Filtrado de artefactos del tomógrafo fuera de límites físicos.
   - Optimización de memoria (`float64` a `float32`).
4. **Transformación de Variables (Data Transformation):**
   - Ventaneo Tomográfico Pulmonar (*Lung Windowing*: [-1000, 400] HU).
   - Normalización Min-Max al rango [0.0, 1.0] para estabilidad de gradiente en Deep Learning.
   - Adaptación de máscaras a formato de competencia de 2 canales (`(N, 512, 512, 2)`).
   - Partición estratificada Train / Validation balanceada por patología COVID-19.
5. **Verificación y Visualización de Resultados:** Gráficos comparativos y generador de envíos Kaggle (`submission.csv`).


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Permitir imports del paquete src
sys.path.append(os.path.abspath(".."))

from src.etl.extract import load_dataset, validate_dataset_shapes
from src.etl.clean import resolve_mask_overlaps, filter_extreme_hu_artifacts, cast_to_efficient_dtype
from src.etl.transform import apply_hu_window, normalize_min_max, transform_masks_for_competition
from src.utils.metrics_helpers import pixel_wise_f1_score, format_kaggle_submission

print("Librerías del pipeline importadas con éxito.")


## 1. Extracción y Verificación Dimensional de Datos Crudos
Se cargan los arreglos volumétricos `.npy` originales inspeccionando su forma y tipo de dato.


In [ ]:
raw_files = {
    "images_medseg": "../images_medseg.npy",
    "masks_medseg": "../masks_medseg.npy",
    "images_radiopedia": "../images_radiopedia.npy",
    "masks_radiopedia": "../masks_radiopedia.npy",
    "test_images_medseg": "../test_images_medseg.npy",
}

for name, path in raw_files.items():
    if os.path.exists(path):
        arr = np.load(path, mmap_mode='r')
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"{name:20s} | Shape: {str(arr.shape):20s} | Dtype: {str(arr.dtype):8s} | Tamaño: {size_mb:8.2f} MB | Min: {np.min(arr):8.2f} | Max: {np.max(arr):8.2f}")


## 2. Limpieza de Datos (Data Cleaning)

### Hallazgo Crítico: Colisión Multietiqueta en Máscaras
En las máscaras originales de 4 canales, existen píxeles catalogados como **Fondo (Canal 3)** que al mismo tiempo fueron etiquetados como **Vidrio Deslustrado (Canal 0)** o **Consolidación (Canal 1)**:
- `masks_medseg.npy`: 510 píxeles solapados.
- `masks_radiopedia.npy`: 22,027 píxeles solapados.

**Decisión de Limpieza:** Jerarquía de prioridad anatómica. Si un vóxel contiene lesión o pulmón, se impone `Background = ~(ch0 | ch1 | ch2)`.


In [ ]:
# Carga de máscaras originales
mask_m = np.load("../masks_medseg.npy", mmap_mode='r')
mask_r = np.load("../masks_radiopedia.npy", mmap_mode='r')

# Detección de solapamiento
overlap_m_before = np.sum((mask_m[..., 0] | mask_m[..., 1]) & mask_m[..., 3])
overlap_r_before = np.sum((mask_r[..., 0] | mask_r[..., 1]) & mask_r[..., 3])
print(f"Colisiones Fondo vs Lesión ANTES:")
print(f"  - Medseg:     {overlap_m_before:,} píxeles")
print(f"  - Radiopedia: {overlap_r_before:,} píxeles")

# Aplicación de limpieza
mask_m_clean, info_m = resolve_mask_overlaps(mask_m)
mask_r_clean, info_r = resolve_mask_overlaps(mask_r)

overlap_m_after = np.sum((mask_m_clean[..., 0] | mask_m_clean[..., 1]) & mask_m_clean[..., 3])
overlap_r_after = np.sum((mask_r_clean[..., 0] | mask_r_clean[..., 1]) & mask_r_clean[..., 3])
print(f"Colisiones Fondo vs Lesión DESPUÉS de la limpieza:")
print(f"  - Medseg:     {overlap_m_after} píxeles (100% corregido)")
print(f"  - Radiopedia: {overlap_r_after} píxeles (100% corregido)")


## 3. Transformación de Variables (Data Transformation)

1. **Ventaneo Hounsfield Pulmonar:**
   - Rango: $[-1000, 400]$ HU (Filtra artefactos de gantry a -1600 HU y metal/hueso denso > 400 HU).
2. **Normalización Min-Max:**
   - Escalamiento al rango continuo $[0.0, 1.0]$.
3. **Máscaras para Competencia Kaggle:**
   - Selección de los canales patológicos `[0, 1]` con salida shape `(N, 512, 512, 2)` en `uint8`.


In [ ]:
# Demostración de transformación en Medseg
img_m = np.load("../images_medseg.npy", mmap_mode='r')
img_m_win = apply_hu_window(img_m, hu_min=-1000.0, hu_max=400.0)
img_m_norm = normalize_min_max(img_m_win, min_val=-1000.0, max_val=400.0)
target_masks_2ch = transform_masks_for_competition(mask_m_clean)

print(f"Imágenes transformadas: Shape={img_m_norm.shape}, Dtype={img_m_norm.dtype}, Rango=[{img_m_norm.min():.2f}, {img_m_norm.max():.2f}]")
print(f"Máscaras competencia:  Shape={target_masks_2ch.shape}, Dtype={target_masks_2ch.dtype}, Clases={np.unique(target_masks_2ch)}")


## 4. Visualización de Resultados y Calidad
Visualizamos un corte representativo de COVID-19 con la sobreposición de vidrio deslustrado (verde) y consolidación (rojo).


In [ ]:
slice_idx = 7  # Corte con marcada presencia de lesiones
ct_slice = img_m_norm[slice_idx, ..., 0]
m_gg = target_masks_2ch[slice_idx, ..., 0] > 0
m_cons = target_masks_2ch[slice_idx, ..., 1] > 0

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(ct_slice, cmap='bone')
axes[0].set_title('Tomografía Ventaneada [0, 1]', fontsize=12, fontweight='bold')
axes[0].axis('off')

# Overlay
rgb = np.repeat(ct_slice[..., None], 3, axis=-1)
rgb[m_gg, 1] = 0.9   # Verde para Ground Glass
rgb[m_cons, 0] = 1.0 # Rojo para Consolidaciones

axes[1].imshow(rgb)
axes[1].set_title('Target Kaggle (Verde: GG | Rojo: Cons.)', fontsize=12, fontweight='bold')
axes[1].axis('off')

# Mapa de clases
axes[2].imshow(m_gg.astype(int) + 2 * m_cons.astype(int), cmap='viridis')
axes[2].set_title('Mapa de Clases de Lesión', fontsize=12, fontweight='bold')
axes[2].axis('off')

plt.tight_layout()
plt.show()


## 5. Metadatos Tabulares y Partición Estratificada Train / Validation
Leemos el resumen estadístico de los 929 cortes generados por el pipeline ETL.


In [ ]:
meta_df = pd.read_csv('../reports/slice_metadata.csv')
print(f"Total de cortes procesados: {len(meta_df)}")
print("
Distribución por partición y presencia de COVID-19:")
print(pd.crosstab(meta_df['split'], meta_df['is_covid_positive'], margins=True))
print("
Primeros registros de metadatos:")
meta_df[['slice_idx', 'dataset', 'is_covid_positive', 'px_ground_glass', 'px_consolidation', 'split']].head(8)


## 6. Generación de Envíos para Kaggle (Submission Helper)
El siguiente bloque demuestra cómo convertir una predicción de `(10, 512, 512, 2)` al formato exacto requerido por Kaggle:


In [ ]:
# Demostración con predicción de prueba
dummy_test_preds = np.zeros((10, 512, 512, 2), dtype=np.uint8)
# Simulamos una pequeña detección
dummy_test_preds[0, 200:250, 200:250, 0] = 1

sub_df = format_kaggle_submission(dummy_test_preds, output_csv_path='../reports/sample_submission.csv')
print(sub_df.head(5))
print(f"Total de filas generadas: {len(sub_df):,}")
